# Testing Area for Nico 

## Description
**TASK**

Implement a regression tree algorithm and a random forest algorithm (based on 
the implemented regression tree algorithm) for predicting numeric values– You can find various implementations for these algorithms. However, we can also apply our own ideas for splitting of instances
+ We should implement these algorithms from scratch (not using any part of existing code)
+ We can use existing code/functions for general parts like: Code for reading the input and testing the algorithm (cross- validation, performance metrics for regression...) 

**COMPARISON**

Compare the implemented techniques with the existing implementations of regression trees/random forest and one other existing regression techniques (we may use the default parameters for the existing techniques) 

+ Experiment with at least three configurations (number of trees, ...) for random 
forest
+ Using at least two performance metrics for comparison
+ Applying cross-validation

***Conclusions***
+ How efficient are our algorithms?
+ Performance of our algorithms?
+ Other findings


## Code section

In [46]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

from typing import Literal, Dict
from numpy.typing import ArrayLike

In [2]:
DecisionTreeRegressor

sklearn.tree._classes.DecisionTreeRegressor

The principle of building a Regression Tree follows the same approach as the creation of a Classification Tree.

We search for the feature which splits the target feature values most purely, divide the dataset along the values of this descriptive feature and repeat this process for each of the sub datasets until we accomplish a stopping criteria. If we accomplish a stopping criteria, we grow a leaf node.

Most notable difference is when to stop:
If we now consider the property of our new continuously scaled target feature we mention that the third stopping criteria can no longer be used since the target feature values can now take on an infinite number of different values. Consequently, it is most likely that we will not find pure target feature values until there is only one instance left in the dataset.

Long story short, there is in general nothing like pure target feature values.

To address this issue, we will introduce an early stopping criteria that returns the average value of the target feature values left in the dataset if the number of instances in the dataset is e.g. <= 5.

In [ ]:
class RegressionTreeNico():
    '''
     TBD

    '''    

    def __init__(self, max_depth=4) -> None:
        self.max_depth = max_depth

    def _calc_MSE(self, Y_true: ArrayLike, Y_pred: ArrayLike) -> np.float64:
        '''
        Calculates mean squared error

        Returns:
            MSE of given features
        
        '''
        return np.square(np.subtract(Y_true, Y_pred)).mean()


    def _calc_variance_cat_features(self, data: ArrayLike, target_name: str, which_feature_name: str) -> np.float64:
        '''
        Used to calculate the variance of categorical features to decide which feature to use for the next leaf via minimum variance!

        Parameters
        ----------
        data : ArrayLike
            The data set in which we want to calculate the variance for a given feature
        target_name: str
            Name of the target feature
        which_feature_name: str
            Name of the feature for which we want to calculate the variance.

        Returns
        -------
        np.float64
            The (weighted) variance of the feature in the given data
        '''

        feature_values = np.unique(data[which_feature_name])
        feature_variance = 0

        for value in feature_values:
            subset = data[data[which_feature_name] == value].reset_index()

            # In case we have only one appearance of a value, then we have to make sure it makes 0 instead of inf because of the division / (N - 1)
            if len(subset) <= 1:
                subset_var = 0.0
            else:
                # standard in np.var is divided by N instead of N - 1!
                subset_var = (len(subset)/len(data)) * np.var(subset[target_name], ddof=1)

            feature_variance += subset_var
        
        return feature_variance
    
    def _calc_variance_cont_features(self, data: ArrayLike, target_name: str, which_feature_name: str) -> np.float64:
        '''
        Used to calculate the variance of continous features to decide which feature to use for the next leaf via minimum variance!
        
        Parameters
        ----------
        data : ArrayLike
            The data set in which we want to calculate the variance for a given feature
        target_name: str
            Name of the target feature
        which_feature_name: str
            Name of the feature for which we want to calculate the variance.

        Returns
        -------
        np.float64
            The (weighted) variance of the feature in the given data
        '''   

        # initialize weighted mse
        best_weighted_mse = np.inf
        best_threshold = np.inf
                    
        # sort the data by the feature
        data = data.sort_values(by=which_feature_name)

        # compute the mse at current node
        current_yhat = np.mean(data[target_name].mean())
        current_mse = self._calc_MSE(data[target_name], current_yhat)
        current_mse = np.round(current_mse, 3)

        # iterate over all rows of the SORTED data
        for i in range(1, len(data)):
            # compute average of two consecutive rows to have some threshold start 
            split_val = (data.iloc[i][which_feature_name] + data.iloc[i-1][which_feature_name]) / 2
            split_val = np.round(split_val, 3)

            print(i, split_val)

            # split the data into the two sides
            left_branch = data[data[which_feature_name]<=split_val]
            right_branch = data[data[which_feature_name]>split_val]

            # compute the MSE of both sides
            left_yhat = np.mean(left_branch[target_name]) 
            left_yhat = np.round(left_yhat, 3)
            left_mse = self._calc_MSE(left_branch[target_name], left_yhat) 
            left_mse = np.round(left_mse, 3)

            right_yhat = np.mean(right_branch[target_name]) 
            right_yhat = np.round(right_yhat, 3)
            right_mse = self._calc_MSE(right_branch[target_name], right_yhat) 
            right_mse = np.round(right_mse, 3)

            # compute weighted MSE
            weighted_mse = ((len(left_branch) * left_mse) + (len(right_branch) * right_mse))/len(data)
            weighted_mse = np.round(weighted_mse, 3)

            # update bestbest_weighted_mse
            if weighted_mse <= best_weighted_mse:
                best_weighted_mse = weighted_mse
                best_threshold = split_val

        return best_weighted_mse, best_threshold

    def _calc_entropy(self, target_column: str) -> np.float64:
        '''
        Calculate the entropy of a target column
       
        Parameters
        ----------
        target_column : ArrayLike
            The target column of which we want to calculate the entropy

        Returns
        -------
        np.float64
            The entropy of the given target column    
        '''

        elements, counts = np.unique(target_column, return_counts = True)

        entropy = np.sum([(-counts[i]/np.sum(counts)) * np.log2(counts[i] / np.sum(counts)) for i in range(len(elements))])
        
        return entropy
    
    def _calc_information_gain(self, data: ArrayLike, target_name:str, which_feature_name: str) -> np.float64:
        '''
        Calculate the information gain based on an attribute (with its weighted entropy!) and the target variable entropy
        
        Parameters
        ----------
        data : ArrayLike
            The data set in which we want to calculate the information gain for a given feature
        target_name: str
            Name of the target feature
        which_feature_name: str
            Name of the feature for which we want to calculate the information gain.

        Returns
        -------
        np.float64
            The infomation gain of the given column    
        '''
        target_entropy = self._calc_entropy(data[target_name])

        vals, counts = np.unique(data[which_feature_name], return_counts=True)

        # Calculate the WEIGHTED entropy
        weighted_entropy = np.sum([(counts[i] / np.sum(counts)) * self._calc_entropy(data[data[which_feature_name]==vals[i]].dropna()[target_name]) for i in range(len(vals))])
    
        # Calculate the information gain
        information_gain = target_entropy - weighted_entropy
        
        return information_gain

    def _find_best_split(self, data: pd.DataFrame, categorical_features:list, target_name: str) -> Dict:
        best_feature = None
        best_threshold = None
        best_variance = np.inf
        is_continuous = False

        for feature_name in data.columns:
            if feature_name == target_name:
                continue
            
            elif feature_name in categorical_features:
                variance = self._calc_variance_cat_features(data, target_name, feature_name)
                
                # if improvement: overwrite values
                if variance <= best_variance:
                    best_feature = feature_name
                    best_threshold = None
                    best_variance = variance
                    is_continuous = False
            
            else:
                variance, threshold = self._calc_variance_cont_features(data, target_name, feature_name)
                                
                # if improvement: overwrite values
                if variance <= best_variance:
                    best_feature = feature_name
                    best_threshold = threshold
                    best_variance = variance
                    is_continuous = True
        
        return best_feature, best_threshold, best_variance, is_continuous


    def Classfier(
            self, 
            data: pd.DataFrame, 
            target_name: str, 
            min_instances: int = 2, 
            categorical_features: list = [], 
            max_depth: int = None, 
            originaldata=None, 
            features: list = None,
            parent_target_mean:float = None,
            depth: int = 0):
        '''
        Recursive tree building algorithm:

        

        Parameters
        ----------

        
        Returns
        ----------
        
        '''
        # Stopping criterion for the recurrsion if we require some minimum value for the size/length of a sub dataset
        if len(data) <= int(min_instances):
            return np.mean(data[target_name])
    
        # If the dataset has reached the wished for tdepth, return the mean target feature value of the remaining dataset as before
        elif depth >= max_depth and max_depth != None:
            return np.mean(data[target_name])

        # Now this is the actual "tree growing" part
        else:
            # Find best split
            best_feature, best_threshold, best_score, is_continuous = self._find_best_split(data, target_name)

            features = data.columns
            features.remove(target_name)

            # get the best split
            best_feature, best_threshold, best_variance, is_continuous = self._find_best_split(data, categorical_features, target_name)

            if is_continuous:
                # build the branching
                left_branch = data[data[best_feature] <= best_threshold]
                right_branch = data[data[best_feature] > best_threshold]
                return {
                    "feature": best_feature,
                    "threshold": best_threshold,
                    "left": self.Classifier(left_branch, target_name, min_instances, max_depth, depth+1),
                    "right": self.Classifier(right_branch, target_name, min_instances, max_depth, depth+1)
                }
            
            else:
                branches = {}
                for val in np.unique(data[best_feature]):
                    subset = data[data[best_feature] == val]
                    branches[val] = self.Classifier(subset, target_name, min_instances, max_depth, depth+1)
                return {
                    "feature": best_feature,
                    "branches": branches
                }

        return None



In [ ]:
for i in None:
    print(i)

TypeError: 'NoneType' object is not iterable

In [23]:
# "test" data

# continous example
np.random.seed(0)

a, b, c = 1, 2, 3
n = 100 
x = np.linspace(-10, 10, n)  # feature values from -10 to 10
noise = np.random.normal(0, 10, n)  # some random noise
y = a * x**2 + b * x + c + noise  # quadratic equation with noise

df_continous = pd.DataFrame({'X': x, 'y': y})

# categorical examples
df_small = pd.DataFrame({'Number_of_Bedrooms':[2,2,4,1,3,1,4,2],'Price_of_Sale':[100000,120000,250000,80000,220000,170000,500000,75000]})

df = pd.read_csv("day.csv",usecols=['season','holiday','weekday','weathersit','cnt'])
df_example = df.sample(frac=0.012)



In [ ]:
testTree = RegressionTreeNico()
#testTree._calc_variance_cat_features(df_example, target_name="cnt", which_feature_name="season")
#testTree._calc_variance_cat_features(df_small, target_name="Price_of_Sale", which_feature_name="Number_of_Bedrooms")
#testTree._calc_variance_cont_features(df_continous, target_name="y", which_feature_name="X")
#testTree.Classfier(df_continous, target_name="y")

1 -9.899
2 -9.697
3 -9.495
4 -9.293
5 -9.091
6 -8.889
7 -8.687
8 -8.485
9 -8.283
10 -8.081
11 -7.879
12 -7.677
13 -7.475
14 -7.273
15 -7.071
16 -6.869
17 -6.667
18 -6.465
19 -6.263
20 -6.061
21 -5.859
22 -5.657
23 -5.455
24 -5.253
25 -5.051
26 -4.848
27 -4.646
28 -4.444
29 -4.242
30 -4.04
31 -3.838
32 -3.636
33 -3.434
34 -3.232
35 -3.03
36 -2.828
37 -2.626
38 -2.424
39 -2.222
40 -2.02
41 -1.818
42 -1.616
43 -1.414
44 -1.212
45 -1.01
46 -0.808
47 -0.606
48 -0.404
49 -0.202
50 0.0
51 0.202
52 0.404
53 0.606
54 0.808
55 1.01
56 1.212
57 1.414
58 1.616
59 1.818
60 2.02
61 2.222
62 2.424
63 2.626
64 2.828
65 3.03
66 3.232
67 3.434
68 3.636
69 3.838
70 4.04
71 4.242
72 4.444
73 4.646
74 4.848
75 5.051
76 5.253
77 5.455
78 5.657
79 5.859
80 6.061
81 6.263
82 6.465
83 6.667
84 6.869
85 7.071
86 7.273
87 7.475
88 7.677
89 7.879
90 8.081
91 8.283
92 8.485
93 8.687
94 8.889
95 9.091
96 9.293
97 9.495
98 9.697
99 9.899


np.float64(665.506)

In [38]:
np.sum(testTree.Classfier(df_continous, target_name="y"))

1 -9.899
2 -9.697
3 -9.495
4 -9.293
5 -9.091
6 -8.889
7 -8.687
8 -8.485
9 -8.283
10 -8.081
11 -7.879
12 -7.677
13 -7.475
14 -7.273
15 -7.071
16 -6.869
17 -6.667
18 -6.465
19 -6.263
20 -6.061
21 -5.859
22 -5.657
23 -5.455
24 -5.253
25 -5.051
26 -4.848
27 -4.646
28 -4.444
29 -4.242
30 -4.04
31 -3.838
32 -3.636
33 -3.434
34 -3.232
35 -3.03
36 -2.828
37 -2.626
38 -2.424
39 -2.222
40 -2.02
41 -1.818
42 -1.616
43 -1.414
44 -1.212
45 -1.01
46 -0.808
47 -0.606
48 -0.404
49 -0.202
50 0.0
51 0.202
52 0.404
53 0.606
54 0.808
55 1.01
56 1.212
57 1.414
58 1.616
59 1.818
60 2.02
61 2.222
62 2.424
63 2.626
64 2.828
65 3.03
66 3.232
67 3.434
68 3.636
69 3.838
70 4.04
71 4.242
72 4.444
73 4.646
74 4.848
75 5.051
76 5.253
77 5.455
78 5.657
79 5.859
80 6.061
81 6.263
82 6.465
83 6.667
84 6.869
85 7.071
86 7.273
87 7.475
88 7.677
89 7.879
90 8.081
91 8.283
92 8.485
93 8.687
94 8.889
95 9.091
96 9.293
97 9.495
98 9.697
99 9.899


np.float64(665.506)

In [ ]:
df_example